<a href="https://colab.research.google.com/github/ewilpeers/bible/blob/master/xG_Collab/01Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mēķis: atrast visas vietas kur šis minēts (takā konkordance līdzīgi)

# definīcijas

In [ ]:
class stulbaisGithubSecretsTikParaizsTikDrosh():
    def encode(sl, s: str) -> bytes:
        # Rotate each UTF-8 byte right by 1. Returns bytes (not str) so the
        # payload can't be mangled by text-encoding round-trips.
        result = bytearray()
        for b in s.encode('utf-8'):
            result.append(((b & 1) << 7) | (b >> 1))
        return bytes(result)

    def encode_escaped(sl, s: str) -> str:
        # Returns a Python bytes literal you can paste into source,
        # e.g. b'\x8a\xbc...'
        encoded = sl.encode(s)
        return "b'" + ''.join(f'\\x{b:02x}' for b in encoded) + "'"

    def decode(sl, s) -> str:
        # Accepts bytes (preferred) or a latin-1 str for backwards-compat.
        if isinstance(s, str):
            s = s.encode('latin-1')
        result = bytearray()
        for b in s:
            result.append(((b & 0x80) >> 7) | ((b << 1) & 0xFF))
        return result.decode('utf-8')

    def print_assignment(sl, name: str, s: str) -> None:
        # Prints a ready-to-paste line: NAME = b'\x..\x..'
        print(f"{name} = {sl.encode_escaped(s)}")


gh = stulbaisGithubSecretsTikParaizsTikDrosh()

In [ ]:
import pandas as pd
TOKEN_NT_DATA_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\x21\x29\x9a\x23\xa2\x23\xa4\x18\xb2\x38\x19\x39\xa0\xbc\x1c\x3a\xac\x24\xb2\x35\xaf\xa8\x9c\x18\x9c\xb3\xa1\xb8\xa2\xa2\xb2\x26\xb9\x19\x2d\x31\x3c\x39\x35\xb3\x21\xac\x26\xa1\x3d\x29\xa9\xb2\x2a\xb8\x34\x1c\x29\x32\xb8\xa1\xa4\xa1\x18\xac\xa8\xa8\x3b\x37\x26\xa6\xaa\xa4\x25\x23\xa6\x24\x9b\xa2\x33\x34\xa5\xb1\x2c\x1b'
TOKEN_NT_DATA = gh.decode(TOKEN_NT_DATA_FU_GITHUB)
TOKEN_OT_DATA_FU_GITHUB = b'\xb3\xb4\x3a\x34\xba\x31\xaf\x38\xb0\x3a\xaf\x98\x98\xa0\xaa\x28\xa2\x9a\x9b\xa8\x18\xbc\x32\x9a\xa7\xb0\xb3\xb3\x33\x19\xa9\xb0\x3d\xaf\xa3\x3c\xb4\xa5\x9b\x39\x1a\x9a\x36\x3a\x99\x9a\x2b\xb4\x2c\xb6\x19\xb6\xa7\x3a\xac\xa0\xa5\xa3\x38\x33\x3b\x18\xb9\x38\xb0\x2c\x39\xbb\xb2\xb9\x99\xa1\xb3\xaa\xa7\xa6\xb4\x9a\x2d\x2c\x23\x24\xa8\xa9\xa5\xb4\x3d\x18\xb2\x25\x3d\x32\x39'
TOKEN_OT_DATA = gh.decode(TOKEN_OT_DATA_FU_GITHUB)
NT_REPO_DATA_PATH = "ewilpeers/new-testament-data"
OT_REPO_DATA_PATH = "grauziitisos/ot-west-len-data"

In [ ]:

import requests
import pandas as pd
def download_csv_from_private_repo(repopath, path_in_repo, token, *args, **kwargs):
	url = f"https://api.github.com/repos/{repopath}/contents/{path_in_repo}"
	r = requests.get(url, headers={
		"Authorization": f"token {token}",
		"Accept": "application/vnd.github.v3.raw"
	})
	r.raise_for_status()
	from io import StringIO
	import pandas as pd
	return pd.read_csv(StringIO(r.text), *args, **kwargs)

In [ ]:
import requests
import pandas as pd
from io import BytesIO

def download_csv_from_release(repopath, tag, filename, token=None, *args, **kwargs):
    if token:
        # Private repo: go through the API with auth.
        # First, find the asset ID for this filename within the release.
        api_url = f"https://api.github.com/repos/{repopath}/releases/tags/{tag}"
        meta = requests.get(api_url, headers={
            "Authorization": f"token {token}",
            "Accept": "application/vnd.github+json",
        })
        meta.raise_for_status()
        assets = meta.json()["assets"]
        asset = next((a for a in assets if a["name"] == filename), None)
        if asset is None:
            raise FileNotFoundError(
                f"{filename!r} not found in release {tag!r}. "
                f"Available: {[a['name'] for a in assets]}"
            )

        # Now fetch the asset bytes. octet-stream is the magic header.
        blob = requests.get(asset["url"], headers={
            "Authorization": f"token {token}",
            "Accept": "application/octet-stream",
        })
        blob.raise_for_status()
        return pd.read_csv(BytesIO(blob.content), *args, **kwargs)


    url = f"https://github.com/{repopath}/releases/download/{tag}/{filename}"
    return pd.read_csv(url, *args, **kwargs)


In [ ]:
l65_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "RT65_words.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
l24_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "JTR2024_words.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
hb_df = download_csv_from_release(OT_REPO_DATA_PATH, "data-v1", "biblehub_hb_en_direct.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
strongs_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "greek-enh-lxx-apo-OT-blb.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
abp_strongs_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, "biblehub_LXX_EXT_el_en_direct.csv", TOKEN_OT_DATA, dtype={'strong_num': str})
l1694_df = download_csv_from_private_repo(OT_REPO_DATA_PATH, 'GL1694_words.csv', TOKEN_OT_DATA)


In [ ]:
nt_strongs_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "strongs.csv", TOKEN_NT_DATA)
nt_lv65_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "latvian_full65.csv", TOKEN_NT_DATA)
nt_l24_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "JTR2024_words.csv", TOKEN_NT_DATA, dtype={'strong_num': str})
nt_l1694_df = download_csv_from_private_repo(NT_REPO_DATA_PATH, "GL1694_words.csv", TOKEN_NT_DATA)


In [ ]:
hb_df.head()

In [ ]:
nt_strongs_g = nt_strongs_df.groupby(["book", "chapter", "verse"])
nt_lv_g = nt_lv65_df.groupby(["book", "chapter", "verse"])
nt_l24_g = nt_l24_df.groupby(["book", "chapter", "verse"])
nt_l1694_g = nt_l1694_df.groupby(["book", "chapter", "verse"])

l65_g = l65_df.groupby(["book", "chapter", "verse"])
l24_g = l24_df.groupby(["book", "chapter", "verse"])
hb_g = hb_df.groupby(["book", "chapter", "verse"])
strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
abp_strongs_g = abp_strongs_df.groupby(["book", "chapter", "verse"])
l1694_g = l1694_df.groupby(["book", "chapter", "verse"])

In [ ]:
import urllib.request, zipfile, io, os, pathlib

OT_URL = "https://github.com/ewilpeers/bible/releases/download/data-v0/bible_ot.zip"
NT_URL = "https://github.com/ewilpeers/new-testament/releases/download/data-v0/bible_nt.zip"
OT_DIR = 'bible/ot'
NT_DIR = 'bible/nt'
def fetch_and_extract(url, dest):
    dest = pathlib.Path(dest)
    if dest.exists() and any(dest.iterdir()):
        return dest  # already cached
    dest.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(url) as r:
        zipfile.ZipFile(io.BytesIO(r.read())).extractall(dest)
    return dest

ot_dir = None
if not os.path.exists(OT_DIR):
  ot_dir = fetch_and_extract(OT_URL, OT_DIR)
else:
  ot_dir =  pathlib.Path(OT_DIR)
nt_dir = None
if not os.path.exists(NT_DIR):
  nt_dir = fetch_and_extract(NT_URL, NT_DIR)
else:
  nt_dir = pathlib.Path(NT_DIR)

In [ ]:
import json
records = []
leftovers = []
for book_dir in sorted(nt_dir.iterdir()):
    if not book_dir.is_dir():
        continue
    book = book_dir.name
    for json_file in sorted(
        [f for f in book_dir.glob('*.json') if f.stem.isdigit()],
        key=lambda x: int(x.stem)
    ):
        chapter = int(json_file.stem)
        try:
            verses = json.loads(json_file.read_text(encoding='utf-8'))
        except Exception as e:
            print(f"  skip {json_file}: {e}")
            continue

        for i, verse in enumerate(verses):
            for j, gw in enumerate(verse.get('greek_words', [])):
                records.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'word': j, #vardi ir 0-bazeti neka panti
                    'greek': gw.get('greek', []),
                    'latvian': gw.get('latvian', []),
                })
            leftovers.append(
                {
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_latvian', [])
                }
            )

nt_df = pd.DataFrame(records)
nt_leftovers_df = pd.DataFrame(leftovers)

In [ ]:
records = []
leftovers_lv = []
leftovers_gk = []
for book_dir in sorted(ot_dir.iterdir()):
    if not book_dir.is_dir():# or not book_dir.name=='1_samuel':
        continue
    book = book_dir.name
    for json_file in sorted(
        [f for f in book_dir.glob('*.json') if f.stem.isdigit()], #and f.stem=='20'],
        key=lambda x: int(x.stem)
    ):
        chapter = int(json_file.stem)
        try:
            verses = json.loads(json_file.read_text(encoding='utf-8'))
        except Exception as e:
          #shis uzkars uz paris minutem visu lapu, bet izvadiis arii visu VD :D
            #print(f"  skip {json_file}: {e}")
            continue

        for i, verse in enumerate(verses):
            for j, hw in enumerate(verse.get('hebrew_words', [])):
                print(hw)
                records.append({
                  'book': book,
                  'chapter': chapter,
                  'verse': i+1,
                  'word': j,
                  'hebrew': hw.get('hebrew', []),
                  'greek': hw.get('greek', []),
                  'latvian': hw.get('latvian', [])
                })

            leftovers_lv.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_latvian', [])
                })
            leftovers_gk.append({
                    'book': book,
                    'chapter': chapter,
                    'verse': i+1,
                    'leftover_latvian': verse.get('leftover_greek', [])
                })




ot_df = pd.DataFrame(records)
ot_leftovers_lv_df = pd.DataFrame(leftovers_lv)
ot_leftovers_gk_df = pd.DataFrame(leftovers_gk)

## utils

In [95]:
_LV_TABLE = str.maketrans('āčēģīķļņõŗšūžĀČĒĢĪĶĻŅÕŖŠŪŽ',
                          'acegiklnorsuzACEGIKLNORSUZ')

def strip_latvian_deep(s: str) -> str:
    return s.lower().translate(_LV_TABLE)

In [102]:
class searchExecutorSimple():
  def __init__(s, df, transFunc, case=False, regex=False):
    s.df = df
    s.case = case
    s.regex = regex
    s.transFunc = transFunc
  def search(s, searchStr):
    if s.case:
      return s.df[s.df['form'].apply(s.transFunc).str.contains(searchStr, case=False)]
    else:
      return s.df[s.df['form'].apply(s.transFunc).str.contains(searchStr, case=False)]

In [111]:
def verse_set(df, mapper = lambda x: x):
    return set(zip(df['book'], df['chapter'], df['verse']))

# darba lauks

In [94]:
nt_df.head()

,book,chapter,verse,word,greek,latvian
0,1_corinthians,1,1,0,Παῦλος,[Pāvils]
1,1_corinthians,1,1,1,κλητὸς,[aicināts]
2,1_corinthians,1,1,2,ἀπόστολος,[apustulis]
3,1_corinthians,1,1,3,Χριστοῦ,[Kristus]
4,1_corinthians,1,1,4,Ἰησοῦ,[Jēzus]


In [91]:
ot_df.head()

,book,chapter,verse,word,hebrew,greek,latvian
0,1_chronicles,1,1,0,אָדָ֥ם,[Αδαμ],[Ādams]
1,1_chronicles,1,1,1,שֵׁ֖ת,[Σηθ],[Sets]
2,1_chronicles,1,1,2,אֱנֽוֹשׁ׃,[Ενως],[Ēnošs]
3,1_chronicles,1,2,0,קֵינָ֥ן,[Καιναν],[Kenans]
4,1_chronicles,1,2,1,מַהֲלַלְאֵ֖ל,[Μαλελεηλ],[Mahalaleēls]


# 1. daļa no- vai tieša sakritība, der lieli un mazi burti, neievērot garumzīmju atšķirības

## 1.1. visvienkāršākais, esošs meklējums 1 valodas bībeles konkrētajā izdevumā

In [119]:
SEARCH1='tic'
SEARCH2='cer'
SEARCH3='mīl'

SEARCH = SEARCH1

atgKaSauc = """
l65_df
l24_df
hb_df
strongs_df
abp_strongs_df
l1694_df
"""
#, case=False, regex=False
lv65_ic_tr= searchExecutorSimple(l65_df, strip_latvian_deep, case=False, regex=False)
lv24_ic_tr= searchExecutorSimple(l24_df, strip_latvian_deep, case=False, regex=False)
lv1694_ic_tr= searchExecutorSimple(l1694_df, strip_latvian_deep, case=False, regex=False)


resultats_tic_65_1 = lv65_ic_tr.search(SEARCH)
print(resultats_tic_65_1.head())

resultats_tic_24_1 = lv24_ic_tr.search(SEARCH)
print(resultats_tic_24_1.head())

resultats_tic_1694_1 = lv1694_ic_tr.search(SEARCH.replace('c', 'z'))
print(resultats_tic_1694_1.head())

          book  chapter  verse  word_idx       form
6608   genesis       15      6         2  uzticējās
9496   genesis       20      8        15    noticis
10164  genesis       21     23        12   uzticību
11542  genesis       24     16        11   aizticis
11744  genesis       24     27        20   uzticību
          book  chapter  verse  word_idx      form
6176   genesis       15      6         2    ticēja
10979  genesis       24     27        16  uzticību
14634  genesis       29     13        25   noticis
25536  genesis       45     26        25     ticēt
29724   exodus        4      1         4   neticēs
           book  chapter  verse  word_idx          form
3057   2_samuel       13     35        19      notizzis
4616   2_samuel       15     20        28    Peetizziba
9996   2_samuel       20     19         7  peetizzigeem
11596  2_samuel       23      1        14     (Tizzibâ)
13156  2_samuel        2      6         9    Peetizzibu


In [118]:
l1694_df.query(
    "book=='genesis' & chapter==15 & verse==6"
)

,book,chapter,verse,word_idx,form
173448,genesis,15,6,0,Un
173449,genesis,15,6,1,wiꞥſch
173450,genesis,15,6,2,tizzeja
173451,genesis,15,6,3,eekẜch
173452,genesis,15,6,4,to
173453,genesis,15,6,5,KUNGU
173454,genesis,15,6,6,un
173455,genesis,15,6,7,tas
173456,genesis,15,6,8,peelahgadija
173457,genesis,15,6,9,wiꞥꞥam


In [120]:
resultats_tic_1694_1.query(
    "book=='genesis' & chapter==15 & verse==6"
)

,book,chapter,verse,word_idx,form
173450,genesis,15,6,2,tizzeja


### 1.1.1. salīdzinājums

In [121]:
def printDiffCounts(set1, set2, titleString='',
                    firstName='1.', secondName='2.'):
  only_in_1 =  set1 - set2
  only_in_2 =  set2 - set1
  in_both = set1 & set2
  print(f"{titleString} skaiti: abos: {len(in_both)}\n {firstName}: {len(only_in_65_tic_1)}\n {secondName}: {len(only_in_24_tic_1)}\n")


In [122]:
set_65_tic_1 = verse_set(resultats_tic_65_1)
set_24_tic_1 = verse_set(resultats_tic_24_1)
set_1694_tic_1 = verse_set(resultats_tic_1694_1)



In [123]:
printDiffCounts(set_65_tic_1, set_24_tic_1,
                SEARCH1.capitalize(), '1965', '2024')

Tic skaiti: abos: 145
 1965: 201
 2024: 138



In [124]:
printDiffCounts(set_65_tic_1, set_1694_tic_1,
                SEARCH1.capitalize(), '1965', '1694')

Tic skaiti: abos: 128
 1965: 201
 1694: 138



In [125]:
printDiffCounts(set_24_tic_1, set_1694_tic_1,
                SEARCH1.capitalize(), '2024', '1694')

Tic skaiti: abos: 107
 2024: 201
 1694: 138



Piezīme: Šeit atkal ir moments,<br>
 kad nav taisnvirziena kustība: <br>
**ideja (T)**<br>
(paņemt 3 vērtību vārdus no Pāvila vēstules, ticība, cerība, mīlestība -> rezultāts ar visām vietām kur tas minēts attiecīgajās valodās, meklētās saknes saturošie vārdi iezīmēti)<br>
**situācija (S)**<br>
pie pirmā, apskatot cik atšķirās cik ir kopīgi, ir negaidīti liels atšķirīgo īpatsvars<br>
**hipotēzes:** <br>
0: tas dēļ tā, ka nav mappings (numerācija atšķiras, daži ir apvienoti panti...)<br>
1: nē, tas cita iemesla dēļ (panti atšķiras u.c. iemesli)<br>
**rīcība (A)**<br>
pielikt garo mapping, jo 2024 iet pa taisno pēc hbrw mappinga, 1965 pēc translācijas tabulām piemeklē atbilstošās vietas numuru vai tā daļu<br>
**secinājums**<br>
(pēc pielikšanas); tikmēr glika tic sakne<br>

* Nolēmu tomēr vispirms iziet visus 3 paraugus (sampling)

Metodika tāpati:<br/> 1. iegūst vienu paraugu modernajā rakstā <br/> 2: to pašu pantu apskata Glika <br/> 3: pārveido meklēšanas sakni

In [127]:
resultats_tic_65_1 = lv65_ic_tr.search(SEARCH2)
print(resultats_tic_65_1[:3])

resultats_tic_24_1 = lv24_ic_tr.search(SEARCH2)
print(resultats_tic_24_1[:3])

resultats_tic_1694_1 = lv1694_ic_tr.search(SEARCH2.replace('c', 'z'))
print(resultats_tic_1694_1[:3])

          book  chapter  verse  word_idx       form
3445   genesis        8      1         2  atcerējās
20732  genesis       36     21         1      Ecers
20814  genesis       36     27         3      Ēcera
         book  chapter  verse  word_idx       form
3241  genesis        8      1         2  atcerējās
3870  genesis        9     15         2  atcerēšos
3901  genesis        9     16         8  atcerētos
          book  chapter  verse  word_idx     form
3143  2_samuel       14      1         3  Zervjas
5335  2_samuel       16      9         3  Zerujas
5368  2_samuel       16     10        10  Zerujas


In [129]:
" ".join(l1694_df.query(
    "book=='genesis' & chapter==8 & verse==1"
)["form"])

'UN DEews atgahdajahs Noßs un wiẜẜo Swehru un wiẜẜo lohpu kas ar to Ꞩchꞣirſtâ bija un DEews dewe Wehju wirs Semmes un tee Uhdeni norimmahs'

at**cer**ējās -> at**gād**ājās

In [131]:
no=5
lidz=15
print(resultats_tic_65_1[no:lidz+1])
print(resultats_tic_24_1[no:lidz+1])


          book  chapter  verse  word_idx         form
27842  genesis       46     24         4       Jecers
29404  genesis       49      5        11     cērtamie
29632  genesis       49     18         2         ceru
30282  genesis       50     20         3   iecerējuši
31619   exodus        3     15        34  atcerieties
50247   exodus       34     13         9    nocērtiet
82087  numbers       11      5         1   atceramies
82879  numbers       11     35         7     Hacerotu
82883  numbers       11     35        11     Hacerotā
83212  numbers       12     16         5     Hacerotu
89717  numbers       21     30         8      cerības
          book  chapter  verse  word_idx         form
19359  genesis       36     21         3        Ēcers
19443  genesis       36     27         2        Ēcera
19472  genesis       36     30         3        Ēcers
19912  genesis       37     11         9    atcerējās
21703  genesis       40     14         1     atceries
21861  genesis       40     

In [132]:
" ".join(l1694_df.query(
    "book=='genesis' & chapter==49 & verse==18"
)["form"])

'KUNGS es gaidu us tawu Peſtiẜchanu'

Tagad vai nu jātaisa mapperis no modernās lv uz seno, vai arī var mēģināt caur pirmvalodas saikni? (grieķu vārds kā rādītājs uz -> iespeējamie translācijas ceļu varianti) <br>
vai nu turpinu ar mihl? vai pāreju uz grieķu/ebreju vārda piemeklējumu <br> Mihl ir ātrāks ceļš, un pie tam lielākā no šām ir mīlestība!